# Feature engineering

The EDA told us **what to look at**; here we build simple, readable features, one idea each:
flags for informative missing values, families of emails and devices, amount features, geography.
Every statistic that uses other purchases (usual amount, card code frequency) is computed on the
**train months only**, so the future is never used.

## 1. Load

The columns chosen in the EDA (`eda_selected_features.json`), plus device and browser from the identity
table (left join: every purchase is kept). Purchases in time order.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)   # harmless pandas warning

RAW = Path("../data/raw")

# 1. the columns chosen in the EDA
selected = json.load(open("eda_selected_features.json"))
base = ["TransactionID", "isFraud", "TransactionDT"]
header = pd.read_csv(RAW / "train_transaction.csv", nrows=0).columns
tx = pd.read_csv(RAW / "train_transaction.csv", usecols=[c for c in base + selected if c in header])

# 2. device and browser, from the second file
identity = pd.read_csv(RAW / "train_identity.csv",
                       usecols=["TransactionID", "DeviceType", "DeviceInfo", "id_30", "id_31"])
df = tx.merge(identity, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(identity["TransactionID"]).astype(int)

# 3. in time order
df = df.sort_values("TransactionDT").reset_index(drop=True)

print(df.shape, f"fraud rate {df['isFraud'].mean() * 100:.2f}%")
print("with device info:", round(df["has_identity"].mean() * 100, 1), "%")

## 2. Time and split

Split **in time**, never random:
- **train** = months 0–3 → the model learns here
- **valid** = month 4 → we choose the threshold (from which score a purchase is sent to an analyst)
- **test** = month 5 → the final exam, looked at only once

In [ ]:
DAY, HOUR = 86400, 3600

df["day"] = df["TransactionDT"] // DAY
df["month"] = (df["day"] // 30).clip(upper=5)
df["hour"] = (df["TransactionDT"] // HOUR) % 24
df["weekday"] = df["day"] % 7

df["split"] = np.select(
    [df["month"] <= 3, df["month"] == 4],   # conditions
    ["train", "valid"],                      # value if the condition is true
    "test",                                  # value for everything else
)

df.groupby("split")["isFraud"].agg(purchases="size", fraud_rate="mean")

## 3. Flags for missing values

From the EDA: some missing (or present) values say a lot. Missing values are **not filled**: the model
(LightGBM) handles them; the flags make the information explicit.

In [ ]:
df["addr1_missing"] = df["addr1"].isna().astype(int)
df["dist2_present"] = df["dist2"].notna().astype(int)
df["R_email_present"] = df["R_emaildomain"].notna().astype(int)
df["n_missing"] = df[selected].isna().sum(axis=1)

train = df[df["split"] == "train"]
for col in ["addr1_missing", "dist2_present", "R_email_present", "has_identity"]:
    rates = train.groupby(col)["isFraud"].mean() * 100
    print(col, rates.round(1).to_dict())

**Finding:** all four flags are strong: no billing address 11.2% fraud (vs 2.5%), dist2 present 9.2% (vs 3.1%), receiver email 7.7% (vs 2.1%), device info present 7.3% (vs 2.1%).

## 4. Email families

~60 email domains → 8 families (google, microsoft, yahoo, …): bigger groups, more reliable rates, and a new
domain in the future (e.g. hotmail.co.uk) still gets a known family.

In [ ]:
EMAIL_FAMILIES = {
    "google": ["gmail", "googlemail"],
    "microsoft": ["hotmail", "outlook", "live", "msn"],
    "yahoo": ["yahoo", "ymail", "rocketmail"],
    "apple": ["icloud", "me.com", "mac.com"],
    "aol": ["aol", "aim.com"],
    "anonymous": ["anonymous"],
    "internet_provider": ["att.net", "sbcglobal", "bellsouth", "verizon", "comcast", "cox.net",
                          "charter", "earthlink", "optonline", "frontier", "windstream",
                          "centurylink", "twc", "roadrunner", "embarqmail", "juno", "netzero"],
}

def family(value, families):
    """The family whose name appears in the value; 'missing' if empty; 'other' if none matches."""
    if pd.isna(value):
        return "missing"
    value = str(value).lower()
    for name, keys in families.items():
        if any(k in value for k in keys):
            return name
    return "other"

df["P_email"] = df["P_emaildomain"].map(lambda v: family(v, EMAIL_FAMILIES))
df["R_email"] = df["R_emaildomain"].map(lambda v: family(v, EMAIL_FAMILIES))
df["same_email"] = (df["P_emaildomain"] == df["R_emaildomain"]).astype(int)

train = df[df["split"] == "train"]
t = train.groupby("P_email")["isFraud"].agg(purchases="size", fraud_pct="mean")
t["fraud_pct"] = (t["fraud_pct"] * 100).round(1)
t.sort_values("purchases", ascending=False)

**Finding:** microsoft emails 5.3% and google 4.4% are above the average; internet providers are the safest (1.3%).

## 5. Device, browser and operating system families

Same idea: thousands of device models → a few brands; "chrome 62.0", "chrome 63.0" → chrome.

In [ ]:
DEVICES = {
    "samsung": ["samsung", "sm-", "gt-"],
    "apple": ["ios", "iphone", "ipad"],
    "windows": ["windows"],
    "mac": ["macos"],
    "motorola": ["moto", "xt1"],
    "lg": ["lg-", "lm-"],
    "huawei": ["huawei", "hi6", "ale-"],
}
BROWSERS = {
    "chrome": ["chrome"], "safari": ["safari"], "firefox": ["firefox"], "edge": ["edge"],
    "ie": ["ie "], "samsung": ["samsung"], "opera": ["opera"],
}
SYSTEMS = {
    "windows": ["windows"], "ios": ["ios"], "mac": ["mac"], "android": ["android"], "linux": ["linux"],
}

df["device"] = df["DeviceInfo"].map(lambda v: family(v, DEVICES))
df["browser"] = df["id_31"].map(lambda v: family(v, BROWSERS))
df["system"] = df["id_30"].map(lambda v: family(v, SYSTEMS))

train = df[df["split"] == "train"]
for col in ["device", "browser", "system"]:
    t = train[train["has_identity"] == 1].groupby(col)["isFraud"].agg(purchases="size", fraud_pct="mean")
    t["fraud_pct"] = (t["fraud_pct"] * 100).round(1)
    print(f"\n{col}\n", t.sort_values("purchases", ascending=False))

**Finding:** mobile brands are risky (huawei 19%, motorola 16%, samsung 12%), mac is very safe (1.7%). Browsers: chrome 8.9%, Internet Explorer 1.7%.

## 6. Amount

- `amt_log`: amount on a log scale (10 $, 100 $, 1000 $ → 1, 2, 3)
- `amt_odd_cents`: more than 2 decimals, typical of a currency conversion (foreign card or shop)

In [ ]:
df["amt_log"] = np.log10(df["TransactionAmt"])
df["amt_odd_cents"] = (np.round(df["TransactionAmt"] * 100, 4) % 1 != 0).astype(int)

train = df[df["split"] == "train"]
print(train.groupby("amt_odd_cents")["isFraud"].mean().round(3).to_dict())

**Finding:** odd cents are one of the strongest signals: 11.2% fraud vs 2.6%, more than 4x.

## 7. Amount compared to the usual, and geography

- `amt_vs_usual`: amount ÷ the usual (median) amount of the same card code `card1`, computed on train only
- `foreign_card`: the card is not from the home country (card3 ≠ 150)
- `foreign_address`: the billing address is not in the home country (addr2 ≠ 87)

In [ ]:
train = df[df["split"] == "train"]

# 1. amount compared to the usual amount of the same card code (card1)
usual = train.groupby("card1")["TransactionAmt"].median()      # usual amount per card1, from train only
df["card1_usual_amt"] = df["card1"].map(usual)
df["amt_vs_usual"] = df["TransactionAmt"] / df["card1_usual_amt"]

# 2. geographic areas: foreign card, foreign billing address
df["foreign_card"] = (df["card3"] != 150).astype(int)          # 150 = home country of the card (EDA)
df["foreign_address"] = (df["addr2"] != 87).astype(int)        # 87 = home billing country (EDA)

# check on train: % of frauds
train = df[df["split"] == "train"]
print("amt_vs_usual:")
print((train.groupby(pd.cut(train["amt_vs_usual"], [0, 0.5, 2, 5, 1e9]), observed=True)["isFraud"]
       .mean() * 100).round(1))
for col in ["foreign_card", "foreign_address"]:
    print(col, (train.groupby(col)["isFraud"].mean() * 100).round(1).to_dict())

**Finding:** foreign cards 10.8% (vs 2.5%) and foreign billing addresses 11.1% (vs 2.4%): strong. Amount vs usual is weak (3.1% → 5.0%).

## 8. Save

Text columns become categories; `card1_freq` = how common the card code is (train only). The table goes to
`data/processed/features.parquet`, the list of features to `features.json`.

In [ ]:
card1_counts = df.loc[df["split"] == "train", "card1"].value_counts()
df["card1_freq"] = df["card1"].map(card1_counts).fillna(0)

CATEGORIES = ["ProductCD", "card4", "card6", "P_email", "R_email",
              "DeviceType", "device", "browser", "system"] + [f"M{i}" for i in range(1, 10)]
for c in CATEGORIES:
    df[c] = df[c].astype("category")

DROP = {"isFraud", "TransactionID", "TransactionDT", "day", "month", "split", "card_id",
        "P_emaildomain", "R_emaildomain", "DeviceInfo", "id_30", "id_31"}
FEATURES = [c for c in df.columns if c not in DROP]

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT / "features.parquet", index=False)
json.dump({"features": FEATURES, "categories": CATEGORIES}, open("features.json", "w"), indent=1)
print(f"{len(FEATURES)} features saved")